# Checagens: o modelo está bom demais?

Os resultados das Etapas A e B são altos (MCC ~0,97). Antes de confiar neles,
este notebook testa explicações alternativas para o acerto alto. Nada aqui
altera os dados: tudo roda em memória, só com treino e validação (o teste
continua intocado até a Etapa E).

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

np.random.seed(42)
torch.manual_seed(42)

dados = np.load("../dados_processados/etapa2.npz", allow_pickle=True)
X_todos = dados["X_simples"].astype(np.float32)  # já normalizado com média/desvio do treino (Etapa 2)
y_especie = dados["y_simples"]
pos_treino = dados["pos_treino_simples"]
pos_val = dados["pos_val_simples"]

## Funções reaproveitadas das Etapas A e B

Mesma MLP do A3 (5-8-1, `Tanh`, saída sigmoide, `BCELoss`, Adam lr 0,01,
mini-batch 64, 400 épocas, pesos do mínimo da perda de validação) e as mesmas
métricas à mão do Slides_6.

In [2]:
def cria_mlp(n_entradas, n_ocultos=8):
    return nn.Sequential(
        nn.Linear(n_entradas, n_ocultos), nn.Tanh(),
        nn.Linear(n_ocultos, 1), nn.Sigmoid(),
    )


def treina(modelo, t_Xtr, t_ytr, t_Xva, t_yva, n_epocas=400, semente=42):
    torch.manual_seed(semente)
    otimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)
    criterio = nn.BCELoss()
    loader = DataLoader(TensorDataset(t_Xtr, t_ytr), batch_size=64, shuffle=True)
    melhor, melhores_pesos = float("inf"), None
    for _ in range(n_epocas):
        modelo.train()
        for xb, yb in loader:
            otimizador.zero_grad()
            perda = criterio(modelo(xb), yb)
            perda.backward()
            otimizador.step()
        modelo.eval()
        with torch.no_grad():
            pv = criterio(modelo(t_Xva), t_yva).item()
        if pv < melhor:
            melhor, melhores_pesos = pv, copy.deepcopy(modelo.state_dict())
    modelo.load_state_dict(melhores_pesos)
    return modelo


def matriz_confusao(y_real, y_prob, limiar=0.5):
    y_pred = (y_prob >= limiar).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_real == 1)))
    fp = int(np.sum((y_pred == 1) & (y_real == 0)))
    tn = int(np.sum((y_pred == 0) & (y_real == 0)))
    fn = int(np.sum((y_pred == 0) & (y_real == 1)))
    return tp, fp, tn, fn


def mcc(tp, fp, tn, fn):
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return ((tp * tn) - (fp * fn)) / den if den > 0 else 0.0


def probabilidades(modelo, X):
    modelo.eval()
    with torch.no_grad():
        return modelo(torch.from_numpy(X)).numpy().flatten()

## Checagem 1: rótulos embaralhados

Embaralha os rótulos do treino (e os da validação usados só para escolher a
época), quebrando a relação entre medidas e espécie. A avaliação final usa os
rótulos **verdadeiros** da validação. Se o pipeline estiver limpo, não há o que
aprender e o MCC deve ficar perto de 0. Se ficar alto, a espécie está vazando
por outro caminho (ordem das linhas, normalização, escolha do checkpoint).

Comparação: o mesmo treino com os rótulos certos. Dois problemas: *lavali* ×
resto (Etapa A) e *lavali* × *temminckii* (Etapa B), 3 embaralhamentos cada.

In [3]:
def monta_problema(nome):
    if nome == "lavali × resto":
        tr, va = pos_treino, pos_val
        pos = "Myotis_lavali"
    else:  # lavali × temminckii
        par = ["Myotis_lavali", "Molossops_temminckii"]
        tr = pos_treino[np.isin(y_especie[pos_treino], par)]
        va = pos_val[np.isin(y_especie[pos_val], par)]
        pos = "Molossops_temminckii"
    y = lambda p: (y_especie[p] == pos).astype(np.float32)
    return X_todos[tr], y(tr), X_todos[va], y(va)


T = lambda a: torch.from_numpy(a)
linhas = []
for problema in ["lavali × resto", "lavali × temminckii"]:
    Xtr, ytr, Xva, yva = monta_problema(problema)

    torch.manual_seed(42)
    m = treina(cria_mlp(5), T(Xtr), T(ytr).unsqueeze(1), T(Xva), T(yva).unsqueeze(1))
    linhas.append({"problema": problema, "rótulos": "verdadeiros", "repetição": 0,
                   "MCC na validação": mcc(*matriz_confusao(yva, probabilidades(m, Xva)))})

    for rep in range(3):
        rng = np.random.default_rng(100 + rep)
        ytr_emb = rng.permutation(ytr)  # cópia embaralhada; ytr original não muda
        yva_emb = rng.permutation(yva)  # só para escolher a época
        torch.manual_seed(rep)
        m = treina(cria_mlp(5), T(Xtr), T(ytr_emb).unsqueeze(1), T(Xva), T(yva_emb).unsqueeze(1), semente=rep)
        prob = probabilidades(m, Xva)
        linhas.append({"problema": problema, "rótulos": "embaralhados", "repetição": rep + 1,
                       "MCC na validação": mcc(*matriz_confusao(yva, prob)),
                       "fração prevista positiva": float((prob >= 0.5).mean())})

df_emb = pd.DataFrame(linhas).round(3)
df_emb

,problema,rótulos,repetição,MCC na validação,fração prevista positiva
0,lavali × resto,verdadeiros,0,0.972,NaN
1,lavali × resto,embaralhados,1,0.096,0.977
2,lavali × resto,embaralhados,2,-0.020,1.000
3,lavali × resto,embaralhados,3,0.024,1.000
4,lavali × temminckii,verdadeiros,0,0.963,NaN
5,lavali × temminckii,embaralhados,1,0.000,0.000
6,lavali × temminckii,embaralhados,2,0.000,0.000
7,lavali × temminckii,embaralhados,3,0.000,0.000


## Checagem 2: a régua da Chave (sem rede)

Os especialistas identificam as espécies por faixas de FME (aba Chave). Se o
rótulo foi dado com essas mesmas medidas, a MLP pode estar só reaprendendo a
regra de quem rotulou. Para medir isso, duas réguas **sem aprendizado nosso**,
só com os valores da Chave:

1. **Binário lavali × resto:** prevê *lavali* se a FME está na faixa da Chave
   (49–52 kHz). Para lavali × temminckii a Chave dá ~50 kHz às duas espécies,
   então não há régua possível.
2. **8 espécies, centro mais próximo:** cada gravação recebe a espécie cujo
   valor de FME na Chave está mais perto da FME medida.

Escolhas para transformar a Chave em números: faixa → ponto médio; "≤18" → 18;
*blossevillii* "≈46 (38–45)" → 46 (o valor declarado); *currentium* → FME1 ≈ 30.

E, como comparação dentro da matéria: a mesma MLP do A3 recebendo **só a FME**
(1-8-1), que mostra o máximo que a FME sozinha rende com a fronteira aprendida.

In [4]:
media, desvio = dados["media_simples"], dados["desvio_simples"]
fme_kHz = X_todos[:, 3] * desvio[3] + media[3]  # volta a FME para kHz (coluna 3 = FME)

fme_val = fme_kHz[pos_val]
esp_val = y_especie[pos_val]

# 1) faixa da Chave, lavali × resto
y_val_lav = (esp_val == "Myotis_lavali").astype(np.float32)
prev_faixa = ((fme_val >= 49) & (fme_val <= 52)).astype(np.float32)
mcc_faixa = mcc(*matriz_confusao(y_val_lav, prev_faixa))
print(f"Régua da faixa 49–52 kHz (lavali × resto), MCC na validação: {mcc_faixa:.3f}")

# 2) centro mais próximo, 8 espécies
centros_chave = {
    "Eumops_sp": 18.0, "Molossus_currentium": 30.0, "Molossus_molossus": 34.0,
    "Neoeptesicus_furinalis": 39.0, "Lasiurus_blossevillii": 46.0,
    "Molossops_temminckii": 50.0, "Myotis_lavali": 50.5, "Myotis_nigricans": 53.5,
}
nomes = np.array(list(centros_chave))
centros = np.array(list(centros_chave.values()))
prev_centro = nomes[np.argmin(np.abs(fme_val[:, None] - centros[None, :]), axis=1)]

sens = {e: float((prev_centro[esp_val == e] == e).mean()) for e in nomes}
print("\nRégua do centro mais próximo (8 espécies), sensibilidade por espécie na validação:")
for e, s in sorted(sens.items(), key=lambda kv: kv[1]):
    print(f"  {e:24s} {s:.3f}")
print(f"Acurácia: {(prev_centro == esp_val).mean():.3f}   "
      f"Acurácia balanceada (média das sensibilidades): {np.mean(list(sens.values())):.3f}")

Régua da faixa 49–52 kHz (lavali × resto), MCC na validação: 0.794

Régua do centro mais próximo (8 espécies), sensibilidade por espécie na validação:
  Molossops_temminckii     0.405
  Myotis_lavali            0.408
  Molossus_currentium      0.486
  Molossus_molossus        0.494
  Lasiurus_blossevillii    0.833
  Neoeptesicus_furinalis   0.928
  Eumops_sp                1.000
  Myotis_nigricans         1.000
Acurácia: 0.545   Acurácia balanceada (média das sensibilidades): 0.694


In [5]:
# MLP só com a FME (1-8-1), mesma receita do A3
linhas_fme = []
for problema in ["lavali × resto", "lavali × temminckii"]:
    Xtr, ytr, Xva, yva = monta_problema(problema)
    for colunas, rotulo in [([0, 1, 2, 3, 4], "5 atributos"), ([3], "só FME")]:
        torch.manual_seed(42)
        m = treina(cria_mlp(len(colunas)), T(Xtr[:, colunas].copy()), T(ytr).unsqueeze(1),
                   T(Xva[:, colunas].copy()), T(yva).unsqueeze(1))
        tp, fp, tn, fn = matriz_confusao(yva, probabilidades(m, Xva[:, colunas].copy()))
        linhas_fme.append({"problema": problema, "entradas": rotulo, "MCC": mcc(tp, fp, tn, fn),
                           "sensibilidade": tp / (tp + fn), "precisão": tp / (tp + fp) if tp + fp else 0.0})

linhas_fme.append({"problema": "lavali × resto", "entradas": "régua da Chave (49–52 kHz)", "MCC": mcc_faixa})
pd.DataFrame(linhas_fme).round(3)

,problema,entradas,MCC,sensibilidade,precisão
0,lavali × resto,5 atributos,0.972,0.988,0.986
1,lavali × resto,só FME,0.896,0.989,0.920
2,lavali × temminckii,5 atributos,0.963,0.988,0.943
3,lavali × temminckii,só FME,0.000,0.000,0.000
4,lavali × resto,régua da Chave (49–52 kHz),0.794,NaN,NaN


## Checagem 3: validação cruzada por local (lavali × resto)

Sugestão do professor: deixar um local inteiro de fora do treino, avaliar nele,
e repetir para todos os locais. Mede se o modelo generaliza para um lugar que
nunca viu (a divisão aleatória mistura gravações do mesmo local e da mesma
noite entre treino e validação).

Em cada dobra (um local L):
- **avaliação:** gravações de L que estão no treino ou na validação da Etapa 2
  (o teste continua intocado);
- **treino:** gravações de treino dos outros locais; **escolha da época:**
  validação dos outros locais;
- **normalização refeita** só com o treino da dobra.

Locais com menos de 100 gravações entram juntos numa dobra "outros". Resultado
principal: MCC com as previsões de todas as dobras juntas, comparável ao da
divisão aleatória. Mesma MLP do A3, semente 42.

In [6]:
ctx = pd.read_csv("../dados_processados/contexto_simples.csv")
assert (ctx["Espécie/Sonótipo"].to_numpy() == y_especie).all()  # mesma ordem de X_simples
area = ctx["Área"].fillna("(vazio)").to_numpy()

X_kHz = X_todos * dados["desvio_simples"] + dados["media_simples"]  # desfaz a normalização da Etapa 2
y_lav = (y_especie == "Myotis_lavali").astype(np.float32)

pos_tv = np.concatenate([pos_treino, pos_val])
tamanho = pd.Series(area[pos_tv]).value_counts()
grandes = tamanho[tamanho >= 100].index.tolist()
dobra = np.where(np.isin(area, grandes), area, "outros")
print(f"{len(grandes)} locais com >= 100 gravações + 'outros' ({int(tamanho[tamanho < 100].sum())} gravações)")

20 locais com >= 100 gravações + 'outros' (294 gravações)


In [7]:
prob_lolo = np.full(len(y_especie), np.nan)
for L in grandes + ["outros"]:
    tr = pos_treino[dobra[pos_treino] != L]
    va = pos_val[dobra[pos_val] != L]
    av = pos_tv[dobra[pos_tv] == L]

    media_d, desvio_d = X_kHz[tr].mean(axis=0), X_kHz[tr].std(axis=0)
    norm = lambda p: ((X_kHz[p] - media_d) / desvio_d).astype(np.float32)

    torch.manual_seed(42)
    m = treina(cria_mlp(5), T(norm(tr)), T(y_lav[tr]).unsqueeze(1), T(norm(va)), T(y_lav[va]).unsqueeze(1))
    prob_lolo[av] = probabilidades(m, norm(av))

avaliadas = pos_tv
tp, fp, tn, fn = matriz_confusao(y_lav[avaliadas], prob_lolo[avaliadas])
print(f"Validação cruzada por local — MCC com todas as dobras juntas: {mcc(tp, fp, tn, fn):.3f}")
print(f"  (n = {len(avaliadas)}, TP={tp} FP={fp} TN={tn} FN={fn})")

Validação cruzada por local — MCC com todas as dobras juntas: 0.971
  (n = 11397, TP=6123 FP=96 TN=5109 FN=69)


In [8]:
linhas_local = []
for L in grandes + ["outros"]:
    p = pos_tv[dobra[pos_tv] == L]
    y, pred = y_lav[p], (prob_lolo[p] >= 0.5).astype(int)
    n_pos, n_neg = int(y.sum()), int((1 - y).sum())
    linhas_local.append({
        "local": L, "n": len(p), "% lavali": round(100 * y.mean()),
        "acurácia": (pred == y).mean(),
        "sensibilidade": ((pred == 1) & (y == 1)).sum() / n_pos if n_pos else np.nan,
        "especificidade": ((pred == 0) & (y == 0)).sum() / n_neg if n_neg else np.nan,
    })
pd.DataFrame(linhas_local).sort_values("acurácia").round(3)